In [1]:
import numpy as np
from numba import njit
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import eigs
import multiprocessing as mp
import os
import time

# --- 1. 加载真值 ---
try:
    TRUE_GAMMAS = np.load("riemann_10k_true.npy")
except:
    print("!!! 错误：未找到 riemann_10k_true.npy")
    exit()

# --- 2. 动力学内核 ---
@njit(fastmath=True, nogil=True)
def harvesting_kernel_v7(u_c, k_static, steps, n_bins):
    x = 0.5
    counts = np.zeros((n_bins, n_bins), dtype=np.float64)
    last_bin = 0
    warmup = 2000000 
    
    for i in range(steps + warmup):
        # 内部微观冷却 (保持 ln^2 结构)
        ln_term = np.log(i + 100)
        mu_eff = u_c + k_static / (ln_term**2)
        
        x = 1 - mu_eff * x**2
        
        if x > 1.0: x = 0.999
        elif x < -1.0: x = -0.999
            
        if i > warmup:
            current_bin = int((x + 1.0) / 2.0 * (n_bins - 1))
            if 0 <= current_bin < n_bins:
                counts[last_bin, current_bin] += 1
                last_bin = current_bin
    return counts

# --- 3. 狙击手 Worker ---
def sniper_worker_v7(task):
    seg_idx, start_n, end_n, k_opt = task
    U_C = 1.543689012
    N_BINS = 20000    
    STEPS = 10**8     
    
    SAVE_DIR = "riemann_10k_harvest_pure_V7"
    
    try:
        t0 = time.time()
        counts = harvesting_kernel_v7(U_C, k_opt, STEPS, N_BINS)
        
        P = csr_matrix(counts)
        row_sums = np.array(P.sum(axis=1)).flatten()
        row_sums[row_sums == 0] = 1.0
        P.data /= row_sums[P.indices]
        
        num_targets = end_n - start_n
        calc_k = num_targets * 2 + 100 
        
        v0 = np.ones(N_BINS)
        vals, _ = eigs(P, k=calc_k, which='LM', tol=1e-4, v0=v0)
        
        # 纯正能量态提取
        pure_vals = vals[(np.abs(vals) > 0.4) & (vals.imag > 1e-7)]
        phases = np.sort(np.angle(pure_vals))
        
        if not os.path.exists(SAVE_DIR):
            os.makedirs(SAVE_DIR, exist_ok=True)
        filename = os.path.join(SAVE_DIR, f"seg_{seg_idx}_n_{start_n}_k_{k_opt:.4f}.npy")
        np.save(filename, phases)
        
        return f"DONE | Seg {seg_idx:3} [n={start_n:5}] | k_opt={k_opt:.4f} | {time.time()-t0:.1f}s"
    except Exception as e:
        return f"FAIL | Seg {seg_idx:3} | {str(e)}"

# --- 4. 战役调度 ---
if __name__ == "__main__":
    SAVE_DIR = "riemann_10k_harvest_pure_V7"
    if not os.path.exists(SAVE_DIR):
        os.makedirs(SAVE_DIR)
        
    print(f"="*60)
    print(f"🚀 V7 终极脱水收割：见证大碗坍缩")
    print(f"="*60)
    
    # 🔥 注入算力炼出的真理参数
    K0 = 2.980559
    BETA = 23.890557
    
    tasks = []
    segment_size = 100
    for i in range(0, 10000, segment_size):
        start_n = i
        end_n = i + segment_size
        mid_n = (start_n + end_n) / 2
        if mid_n < 2: mid_n = 2 
        
        # 严格对应 V7 拟合的公式结构
        k_opt = K0 + BETA / np.log(mid_n)
        tasks.append((i//segment_size, start_n, end_n, k_opt))

    BATCH_SIZE = 100
    start_total = time.time()
    
    with mp.Pool(processes=BATCH_SIZE) as pool:
        results = pool.map(sniper_worker_v7, tasks)
        
    for res in results:
        print(f"  {res}")

    print(f"\n✅ V7 真理数据收割完毕！总耗时: {(time.time()-start_total)/60:.2f} 分钟")

🚀 V7 终极脱水收割：见证大碗坍缩
  DONE | Seg   0 [n=    0] | k_opt=9.0875 | 767.5s
  DONE | Seg   1 [n=  100] | k_opt=7.7485 | 766.9s
  DONE | Seg   2 [n=  200] | k_opt=7.3074 | 668.2s
  DONE | Seg   3 [n=  300] | k_opt=7.0589 | 755.4s
  DONE | Seg   4 [n=  400] | k_opt=6.8911 | 635.4s
  DONE | Seg   5 [n=  500] | k_opt=6.7668 | 759.6s
  DONE | Seg   6 [n=  600] | k_opt=6.6691 | 766.7s
  DONE | Seg   7 [n=  700] | k_opt=6.5894 | 734.6s
  DONE | Seg   8 [n=  800] | k_opt=6.5224 | 751.4s
  DONE | Seg   9 [n=  900] | k_opt=6.4649 | 750.4s
  DONE | Seg  10 [n= 1000] | k_opt=6.4148 | 760.7s
  DONE | Seg  11 [n= 1100] | k_opt=6.3705 | 738.4s
  DONE | Seg  12 [n= 1200] | k_opt=6.3308 | 752.8s
  DONE | Seg  13 [n= 1300] | k_opt=6.2951 | 768.0s
  DONE | Seg  14 [n= 1400] | k_opt=6.2625 | 733.6s
  DONE | Seg  15 [n= 1500] | k_opt=6.2327 | 758.8s
  DONE | Seg  16 [n= 1600] | k_opt=6.2053 | 767.0s
  DONE | Seg  17 [n= 1700] | k_opt=6.1799 | 714.9s
  DONE | Seg  18 [n= 1800] | k_opt=6.1563 | 737.5s
  DONE | Seg